# A图集构建方案

每个spu关联一个主图，n个场景图，m个sku图

需要以结构化存储的方式保存这些图片，保存图片路径
```plaintext
{
    "spu_id": 123456,
    "main_image": "path/to/main_image.jpg",
    "scene_images": [
        "path/to/scene_image1.jpg",
        "path/to/scene_image2.jpg",
        ...
    ],
    "sku_images": [
        "path/to/sku_image1.jpg",
        "path/to/sku_image2.jpg",
        ...
    ]
}

文件夹结构：
A_image_dataset/
    ├── spu_123456/
    │   ├── image_id_1.jpg  # 主图
    │   ├── image_id_2.jpg  # 场景图1
    │   ├── image_id_3.jpg  # 场景图2
    │   ├── image_id_4.jpg  # SKU图1
    │   ├── image_id_5.jpg  # SKU图2
    │   └── ...
    ├── spu_789012/
    │   ├── image_id_1.jpg
    │   ├── image_id_2.jpg
    │   ├── image_id_3.jpg
    │   ├── image_id_4.jpg
    │   ├── image_id_5.jpg
    │   └── ...
    └── ...
每个spu的图片信息存储在一个JSON文件中，方便后续处理
```

In [ ]:
from pydantic import BaseModel, HttpUrl
from enum import Enum
from pathlib import Path
from typing import List, Optional
import requests



def get_main_and_scene_images_ids(spu_id: int):
    """
    获得spu的主图和场景图片id列表
    {
        "spu_id": 123456,
        "main_image_id": "image_id_1",
        "scene_image_ids": ["image_id_2", "image_id_3"],
    }
    """
    DOWNLOAD_URL = 'https://erp.baycheer.com/api/fileAccess/getSpuImage'
    params = {
        "app_id": 392013,
        "app_token": 'URC2P3GKZGDPFAAX8M61LQ88NLRRO3T9',
        "spu_id": spu_id,
        "image_type_id": '2524'
    }
    scene_images_id_dict = {}
    scene_images_id_dict['spu_id'] = spu_id
    scene_images_id_dict['scene_image_ids'] = []
    response = requests.get(DOWNLOAD_URL, params=params)
    if response.status_code == 200:
        resp = response.json()
        if resp['code'] == 0:
            data = resp['data']
            for item in data:
                if item['is_cover_image'] == 1:
                    scene_images_id_dict['main_image_id'] = item['product_image_id']
                    continue
                if item['topic'] =='A':
                    scene_images_id_dict['scene_image_ids'].append(item['product_image_id'])
            return scene_images_id_dict
        else:
            print(f"Error in response for spu_id: {spu_id}, message: {resp['msg']}")
            return None

                
def get_sku_images_ids(spu_id: int):
    """
    获得spu的sku图片id列表
    {
        "spu_id": 123456,
        "sku_image_ids": ['image_id_1', 'image_id_2']
        }
    }
    """
    DOWNLOAD_URL = 'https://erp.baycheer.com/api/fileAccess/getSpuImage'
    params = {
        "app_id": 392013,
        "app_token": 'URC2P3GKZGDPFAAX8M61LQ88NLRRO3T9',
        "spu_id": spu_id,
        "image_type_id": '2528'
    }
    sku_images_id_dict = {}
    sku_images_id_dict['spu_id'] = spu_id
    sku_images_id_dict['sku_image_ids'] = []
    response = requests.get(DOWNLOAD_URL, params=params)
    if response.status_code == 200:
        resp = response.json()
        if resp['code'] == 0:
            data = resp['data']
            for item in data:
                sku_images_id_dict['sku_image_ids'].append(item['product_image_id'])
            return sku_images_id_dict
        else:
            print(f"Error in response for spu_id: {spu_id}, message: {resp['msg']}")
            return None

        
    

def get_images_id_with_spu_id(spu_id: int):
    """
    获得spu的所有图片id
    {
        "spu_id": 123456,
        "main_image_id": "image_id_1",
        "scene_image_ids": ["image_id_2", "image_id_3"],
        "sku_image_ids": ['image_id_4', 'image_id_5'...]
    }
    """
    main_and_scene_images_ids_dict = get_main_and_scene_images_ids(spu_id)
    sku_images_ids_dict = get_sku_images_ids(spu_id)
    if main_and_scene_images_ids_dict and sku_images_ids_dict:
        combined_dict = {
            "spu_id": spu_id,
            "main_image_id": main_and_scene_images_ids_dict.get("main_image_id"),
            "scene_image_ids": main_and_scene_images_ids_dict.get("scene_image_ids", []),
            "sku_image_ids": sku_images_ids_dict.get("sku_image_ids", [])
        }
        return combined_dict
    else:
        return None

In [ ]:
spu_id = 21772288
image_ids_dict = get_images_id_with_spu_id(spu_id)
print(image_ids_dict)